# GSE138458 — Exploratory Data Analysis

Whole-blood gene expression from an adult SLE cohort plus healthy controls (GPL10558, Illumina HumanHT-12 v4). This notebook walks through four findings from Day 1 that shape every modelling decision later in the project — a naive read of the dataset ("307 SLE vs. 23 controls, stratify and go") misses all of them.

Reusable computation (data loading, cohort/PCA/batch summaries) lives in [`biomedical_ml.eda`](../src/biomedical_ml/eda.py) and [`biomedical_ml.preprocessing`](../src/biomedical_ml/preprocessing.py) — imported and unit-tested. This notebook is the visualization and narrative layer on top of it.

In [ ]:
from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from biomedical_ml.config import FIGURES_DIR, RESULTS_DIR, SEED, ensure_dirs, set_seed
from biomedical_ml.data import load_series_matrix
from biomedical_ml.eda import (
    batch_confound_report,
    cohort_summary,
    correlate_pcs_with_batch,
    pca_embedding,
    probe_variance,
)
from biomedical_ml.preprocessing import build_dataset, drop_empty_samples
from biomedical_ml.splits import cv_splitter

set_seed()
ensure_dirs()
sns.set_theme(style="whitegrid", font_scale=0.9)
%matplotlib inline

CASE_COLOUR = "#c44e52"
CONTROL_COLOUR = "#4c72b0"
PALETTE = {"SLE": CASE_COLOUR, "Control": CONTROL_COLOUR}

A few small plotting helpers, kept local to this notebook since they exist to make *this* walkthrough readable rather than to be reused or tested elsewhere.

In [ ]:
def plot_subject_structure(summary: dict) -> plt.Figure:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    counts = [summary["n_control_samples"], summary["n_sle_samples"]]
    subjects = [summary["n_control_subjects"], summary["n_sle_subjects"]]
    x = np.arange(2)
    axes[0].bar(x - 0.2, counts, 0.4, color=[CONTROL_COLOUR, CASE_COLOUR])
    axes[0].bar(x + 0.2, subjects, 0.4, color=[CONTROL_COLOUR, CASE_COLOUR], alpha=0.5)
    axes[0].set_xticks(x, ["Control", "SLE"])
    axes[0].set_title("Class balance:\nsamples vs. subjects")
    axes[0].set_ylabel("count")
    axes[0].legend(
        handles=[
            Patch(facecolor="0.35", label="samples"),
            Patch(facecolor="0.35", alpha=0.5, label="subjects"),
        ],
        frameon=False,
    )
    for xi, (c, s) in enumerate(zip(counts, subjects)):
        axes[0].text(xi - 0.2, c, str(c), ha="center", va="bottom")
        axes[0].text(xi + 0.2, s, str(s), ha="center", va="bottom")

    per_subject = summary["samples_per_subject"]
    axes[1].bar(list(per_subject), list(per_subject.values()), color="#55a868")
    axes[1].set_xticks(list(per_subject))
    axes[1].set_xlabel("samples contributed")
    axes[1].set_ylabel("subjects")
    axes[1].set_title("Repeated measures:\nvisits per subject")
    for k, v in per_subject.items():
        axes[1].text(k, v, str(v), ha="center", va="bottom")

    fig.tight_layout()
    return fig


def plot_expression_distributions(dataset) -> plt.Figure:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    rng = np.random.default_rng(SEED)
    sample_ids = rng.choice(dataset.X.index, size=min(40, len(dataset.X)), replace=False)
    for sample_id in sample_ids:
        colour = CASE_COLOUR if dataset.y.loc[sample_id] == 1 else CONTROL_COLOUR
        axes[0].hist(dataset.X.loc[sample_id], bins=120, histtype="step",
                     density=True, alpha=0.35, color=colour, lw=0.8)
    axes[0].set_xlabel("log2 expression")
    axes[0].set_ylabel("density")
    axes[0].set_title(f"Per-sample intensity distributions\n({len(sample_ids)} random samples)")

    medians = dataset.X.median(axis=1)
    label = dataset.y.map({0: "Control", 1: "SLE"})
    sns.boxplot(x=label, y=medians, hue=label, ax=axes[1], palette=PALETTE, legend=False,
                order=["Control", "SLE"])
    sns.stripplot(x=label, y=medians, ax=axes[1], color="0.25", size=3, alpha=0.5,
                  order=["Control", "SLE"])
    axes[1].set_xlabel("")
    axes[1].set_ylabel("median log2 expression")
    axes[1].set_title("Sample medians are aligned\n(series is already normalised)")

    fig.tight_layout()
    return fig


def plot_pca(scores, variance_ratio, dataset) -> plt.Figure:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    label = dataset.y.map({0: "Control", 1: "SLE"})
    for name, colour in PALETTE.items():
        mask = (label == name).to_numpy()
        axes[0].scatter(scores.loc[mask, "PC1"], scores.loc[mask, "PC2"], s=22, alpha=0.75,
                        c=colour, label=f"{name} (n={int(mask.sum())})",
                        edgecolor="white", linewidth=0.3)
    axes[0].set_title("PC1-PC2 by disease status")
    axes[0].legend(frameon=False)

    chips = dataset.metadata["chip_id"].astype("category").cat.codes
    axes[1].scatter(scores["PC1"], scores["PC2"], s=22, alpha=0.8, c=chips, cmap="tab20",
                    edgecolor="white", linewidth=0.3)
    axes[1].set_title("PC1-PC2 by BeadChip\n(no obvious batch clustering)")

    for ax in axes[:2]:
        ax.set_xlabel(f"PC1 ({variance_ratio[0]:.1%} var)")
        ax.set_ylabel(f"PC2 ({variance_ratio[1]:.1%} var)")

    axes[2].bar(range(1, len(variance_ratio) + 1), variance_ratio * 100, color="#8172b2")
    axes[2].set_xlabel("component")
    axes[2].set_ylabel("% variance explained")
    axes[2].set_title("Scree plot")
    axes[2].set_xticks(range(1, len(variance_ratio) + 1))

    fig.tight_layout()
    return fig


def plot_probe_variance(variance) -> plt.Figure:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(np.log10(variance.to_numpy() + 1e-12), bins=120, color="#937860")
    axes[0].set_xlabel("log10 variance across samples")
    axes[0].set_ylabel("probes")
    axes[0].set_title("Most probes barely vary\n(feature selection has room to work)")

    cumulative = variance.cumsum() / variance.sum()
    axes[1].plot(range(1, len(cumulative) + 1), cumulative.to_numpy(), color="#937860")
    axes[1].set_xscale("log")
    axes[1].set_xlabel("top-N probes by variance")
    axes[1].set_ylabel("cumulative share of total variance")
    axes[1].set_title("Variance concentrates in a small probe subset")
    for n in (500, 2000, 10000):
        if n <= len(cumulative):
            axes[1].axvline(n, ls="--", lw=0.8, color="0.6")
            axes[1].text(n, 0.05, f" {n}", fontsize=8, color="0.4")

    fig.tight_layout()
    return fig

## Finding 1 — GEO ships 336 samples, but only 330 carry data

GSE138458's series matrix lists 336 samples. Six of them are placeholders: `data_row_count == 0` in the metadata, and their expression columns are entirely missing. This is a data-quality step, not a modelling choice, so it happens once, up front, in [`preprocessing.drop_empty_samples`](../src/biomedical_ml/preprocessing.py) — before any split, selection, or scaling touches the matrix.

In [ ]:
expression_raw, metadata_raw = load_series_matrix()
print(f"as published:  {expression_raw.shape[1]} samples, {expression_raw.shape[0]} probes")

empty = expression_raw.columns[expression_raw.isna().all()]
print(f"empty samples: {len(empty)}  {list(empty)}")

expression_clean, metadata_clean = drop_empty_samples(expression_raw, metadata_raw)
print(f"after cleanup: {expression_clean.shape[1]} samples")
print()
print(metadata_clean["case_control"].value_counts())

Dropping the six empty samples lands exactly on the cohort described in the project brief: **307 SLE, 23 healthy control**. Good sign that this is the intended cleanup, not an artefact of how the series matrix happens to be formatted.

Everything below uses [`preprocessing.build_dataset`](../src/biomedical_ml/preprocessing.py), which applies this same cleanup, restricts to annotated probes, and attaches subject and batch metadata.

In [ ]:
dataset = build_dataset(annotated_only=True)
print(dataset.summary())

## Finding 2 — 330 samples come from only 218 subjects

The `subject_id` field reveals repeat visits: 102 patients were sampled twice and 5 patients three times. Splitting at the *sample* level — which a plain stratified split would do — can put two visits from the same patient on opposite sides of a train/test boundary. The model would then get a free look at that patient's baseline expression before being asked to classify their held-out sample, which inflates the apparent performance.

Every split in this project is therefore grouped on `subject_id` **and** stratified on the label (see [`biomedical_ml.splits`](../src/biomedical_ml/splits.py)).

In [ ]:
summary = cohort_summary(dataset)
fig = plot_subject_structure(summary)
fig.savefig(FIGURES_DIR / "cohort_structure.png", dpi=150, bbox_inches="tight")
plt.show()

How much does grouping actually matter here? We can measure it directly: fit the same pipeline — top-2000 `SelectKBest` features, standardize, L2 logistic regression — under a naive sample-level `StratifiedKFold` and under a subject-grouped `StratifiedGroupKFold`, and compare.

In [ ]:
X, y, groups = dataset.X.values, dataset.y.values, dataset.groups.values

pipe = Pipeline([
    ("select", SelectKBest(f_classif, k=2000)),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
])

naive = cross_val_score(
    pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc"
)
grouped = cross_val_score(
    pipe, X, y, cv=cv_splitter(5, SEED), groups=groups, scoring="roc_auc"
)

print(f"naive   StratifiedKFold       ROC-AUC {naive.mean():.3f} +/- {naive.std():.3f}  {np.round(naive, 3)}")
print(f"grouped StratifiedGroupKFold  ROC-AUC {grouped.mean():.3f} +/- {grouped.std():.3f}  {np.round(grouped, 3)}")
print(f"inflation from ignoring subject_id: {naive.mean() - grouped.mean():+.3f} AUC")

Honest result: the leakage inflation is small (+0.002 AUC) — the case/control signal is strong enough here to sit near a ceiling either way. Grouping is kept anyway: it costs nothing, it is the defensible default, and there is no guarantee it stays this negligible once we start doing anything more flexible than L2 logistic regression (an autoencoder, in particular, has far more capacity to memorize a subject's fingerprint).

## Finding 3 — 22 control subjects is the real sample-size constraint

At roughly 13:1 case:control imbalance, a 5-fold split leaves only 4–5 control *subjects* per fold. That is few enough for fold-to-fold variance to swamp genuine model differences. We can see this directly from the grouped folds computed above.

In [ ]:
fold_report = []
for fold, (train_idx, test_idx) in enumerate(cv_splitter(5, SEED).split(X, y, groups)):
    fold_report.append({
        "fold": fold,
        "test_samples": len(test_idx),
        "test_control_subjects": pd.Series(groups[test_idx])[y[test_idx] == 0].nunique(),
        "roc_auc": grouped[fold],
    })
fold_report = pd.DataFrame(fold_report).set_index("fold")
print(fold_report.round(3).to_string())
print(f"\nROC-AUC range across folds: {fold_report['roc_auc'].min():.3f} - {fold_report['roc_auc'].max():.3f}")

A near-0.2 AUC swing between folds, on one model with one fixed configuration, confirms that a single 5-fold split is too noisy to trust as a headline number here. From Day 2 onward, reported metrics are averaged over **repeated** grouped, stratified CV (`splits.repeated_cv_splits`), never a single split — and never plain accuracy, which would be trivially dominated by the majority class.

## Finding 4 — global variance does not track disease status

PCA on the full annotated matrix is the honest linear baseline: does the dominant structure in the data separate SLE from control at all, without ever using the label? Probes are centred but not scaled to unit variance — on log2 array data, scaling would inflate low-expression probes whose variance is mostly measurement noise.

In [ ]:
scores, variance_ratio = pca_embedding(dataset.X, n_components=10)
fig = plot_pca(scores, variance_ratio, dataset)
fig.savefig(FIGURES_DIR / "pca_overview.png", dpi=150, bbox_inches="tight")
plt.show()

The middle panel raises an obvious question: could any apparent structure just be BeadChip batch effects? The `description` field encodes each sample's chip barcode, which lets us check directly.

In [ ]:
batch = batch_confound_report(dataset.metadata, dataset.y)
print(
    f"{batch['n_chips']} BeadChips total; {batch['n_chips_with_control']} carry a control "
    f"(max {batch['max_controls_on_one_chip']} per chip); chips_are_mixed={batch['chips_are_mixed']}"
)

batch_pcs = correlate_pcs_with_batch(scores, dataset.metadata)
print("\nPC vs. BeadChip association (one-way ANOVA):")
print(batch_pcs.round(4).to_string())

The 23 controls sit on 23 *different* chips, one per chip alongside 11 SLE cases — they were deliberately spread across batches, not clustered on a few. Chip explains almost none of PC1 (η² = 0.06, p = 0.82). Whatever signal we find is not a repackaged batch artefact.

But the left panel above answers the actual question: **PCA does not separate the classes.** Controls sit inside the SLE cloud. So what *is* PC1–PC2 capturing, if not diagnosis?

In [ ]:
variance = probe_variance(dataset.X)
top_genes = dataset.annotation.loc[variance.index[:25], "gene_symbol"].dropna().tolist()
print("Top 25 highest-variance genes:")
print(top_genes)

fig = plot_probe_variance(variance)
fig.savefig(FIGURES_DIR / "probe_variance.png", dpi=150, bbox_inches="tight")
plt.show()

This is a useful biological sanity check as much as an EDA result. The top-variance genes are **IFI27, IFI44L, RSAD2, IFIT1, OASL, OAS1** — the type-I interferon signature that is the textbook hallmark of SLE — plus HLA-DRB1/DRB5 (immune), globin genes HBG1/HBG2/ALAS2 (ordinary red-cell carryover in whole blood), and RPS4Y1 (sex).

The interferon axis is exactly what the original GSE138458 study clustered patients on — but it varies *within* the SLE group (some patients have a strong IFN signature, others don't), rather than cleanly separating SLE from healthy. That is why it dominates global variance without showing up as class separation in PC1–PC2: **the largest axis of variation in this dataset is disease heterogeneity, not disease presence.**

For completeness, one more check: is the series properly normalised, so that expression differences reflect biology rather than array-to-array scaling?

In [ ]:
fig = plot_expression_distributions(dataset)
fig.savefig(FIGURES_DIR / "expression_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

Sample medians line up tightly across both groups — GEO ships this series already background-corrected, log2-transformed and normalised (values span roughly 6.7–16.4). No further normalisation is applied downstream; doing so would be cargo-culting a step GEO already did.

## Summary — what this means for Day 2 onward

1. **Clean to 330 samples / 307 SLE / 23 control** before anything else — six placeholder samples are empty.
2. **Group every split on `subject_id`.** Measured leakage from skipping this is small on a linear baseline (+0.002 AUC) but the grouping is free, so it stays as the default regardless of model.
3. **Average over repeated grouped CV, never a single split, never plain accuracy.** 22 control subjects makes single-split fold variance (~0.2 AUC swing here) too large to trust.
4. **PCA is the baseline the autoencoder has to beat.** Global unsupervised variance tracks interferon-signature heterogeneity within the SLE group, not case/control status — confirmed not to be a BeadChip batch artefact. The open question for Day 3–4: does a non-linear latent space recover a disease axis that the top-variance linear directions miss?

The cell below regenerates `results/eda_summary.json` for downstream reference.

In [ ]:
summary["batch"] = batch
summary["pc_variance_ratio"] = [round(float(v), 5) for v in variance_ratio]
summary["pc_vs_chip_eta_squared"] = {
    str(k): round(float(v), 4) for k, v in batch_pcs["eta_squared"].items()
}
summary["top_variable_genes"] = top_genes
summary["subject_grouping_leak_auc"] = round(float(naive.mean() - grouped.mean()), 4)
summary["grouped_cv_auc_range"] = [
    round(float(fold_report["roc_auc"].min()), 3),
    round(float(fold_report["roc_auc"].max()), 3),
]

summary_path = RESULTS_DIR / "eda_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"wrote {summary_path}")